# Library

In [2]:
import scipy.stats as st
import pandas as pd
import numpy as np

from gtda.diagrams import Amplitude, PersistenceLandscape, BettiCurve
from gtda.homology import VietorisRipsPersistence

from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.svm import SVC

from tqdm.notebook import tqdm_notebook

from TFE import *

import path

from pathlib import Path

RESULTS_DIR = Path("Classification_Result")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Function

In [ ]:
def evaluate_model(Model, Features, labels, parameters = 'None', CV = 5, repeat = 10, seed = 42):
    
    fs = SelectPercentile(score_func=f_classif)
    
    
    if Model == 'RF_MVPA':
        
        model = RandomForestClassifier(random_state=seed, n_estimators=300, n_jobs=-1)
        pipe = Pipeline(steps=[("Reduction", "passthrough"),  ("Model", model)])
        parameters = [dict(Model__max_depth =  [2, 3, 5], Model__min_samples_leaf =  [3, 5, 10])]
        
        
    elif Model == 'RF_FC':
        
        model = RandomForestClassifier(random_state=seed, n_estimators=300, n_jobs=-1)
        pipe = Pipeline(steps=[("FS", fs),  ("Reduction", "passthrough"),  ("Model", model)])
        parameters = [dict(Model__max_depth =  [2, 3, 5, None], Model__max_features = ["sqrt", 0.5], Model__min_samples_leaf =  [2, 5, 10], FS__percentile = [1, 2, 3, 5] , Reduction = [PCA(svd_solver='full')], Reduction__n_components = [0.8 , 0.9, 0.95])]
        
        
    elif Model == 'RF_TDA':
        
        model = RandomForestClassifier(random_state=seed, n_estimators=300, n_jobs=-1)
        pipe = Pipeline(steps=[("FS", fs),  ("Reduction", "passthrough"),  ("Model", model)])
        parameters = [dict(Model__max_depth =  [2, 3, 5, 7], Model__min_samples_leaf =  [3, 5, 10, 20], FS__percentile = [i for i in range(5, 65, 5)] , Reduction = [PCA(svd_solver='full')], Reduction__n_components = [0.7, 0.8 , 0.9, 0.99])] 
        
        
    elif Model == 'SVM':
        
        model = SVC(random_state=seed)
        pipe = Pipeline([("FS", fs), ("Scaler", StandardScaler()), ("Reduction", "passthrough"), ("Model", model)])

        parameters = [

            # PCA
            { "Model__C": [0.01, 0.1, 1, 10, 100], "Model__kernel": ["linear", "rbf"], "Model__gamma": ["scale", 0.001, 0.01, 0.1], "FS__percentile": [5, 10, 15, 20, 30, 40, 50, 60], "Reduction": [PCA(svd_solver="full")], "Reduction__n_components": [0.7, 0.8, 0.9, 0.95, 0.99] },

            # LDA
            { "Model__C": [0.01, 0.1, 1, 10, 100], "Model__kernel": ["linear", "rbf"], "Model__gamma": ["scale", 0.001, 0.01, 0.1], "FS__percentile": [5, 10, 15, 20, 30, 40, 50, 60], "Reduction": [LinearDiscriminantAnalysis(n_components=1)] },

            # No dimensionality reduction
            { "Model__C": [0.01, 0.1, 1, 10, 100], "Model__kernel": ["linear", "rbf"], "Model__gamma": ["scale", 0.001, 0.01, 0.1], "FS__percentile": [5, 10, 15, 20, 30, 40, 50, 60] }
        ]
        
    else:
        
        pipe = Model
    
    # Hyperparameter optimization is performed using stratified cross-validation.
    # Feature selection and dimensionality reduction are included in the pipeline
    # to avoid data leakage during cross-validation.
    
    grid = GridSearchCV(pipe, param_grid=parameters, cv=StratifiedKFold(n_splits=CV, shuffle=True, random_state=seed), n_jobs = -1, scoring= 'accuracy')
    
    acc_test_all = np.array([])
    
    
    # Split the data into stratified training and test sets.
    # Multiple iterations are used with different random seeds to assess
    # the stability of classification performance.
    
    for iter in range(repeat):
        
        X_train, X_test, y_train, y_test = train_test_split(Features, labels, test_size=0.15, random_state=iter*2 +3, stratify=labels)
        grid.fit(X_train, y_train)
        acc_test_all = np.append(acc_test_all, grid.score(X_test, y_test))
        
    acc_test_mean = acc_test_all.mean()
    acc_test_std = acc_test_all.std()
    
    if acc_test_mean < 1:
        
        CI = st.t.interval(confidence=0.95, df= repeat-1, loc= acc_test_mean, scale= (acc_test_std / np.sqrt(repeat)))
        CI = np.round(np.array(CI) * 100, 1).reshape(-1)
        if CI[1] > 100:
            
            CI[1] = 100.0
    else:
        
        CI = [1, 1]
        CI = np.round(np.array(CI) * 100, 1).reshape(-1)
        
    return np.round(acc_test_mean * 100 , 1) ,(CI[1] - CI[0]) / 2

In [4]:
def matrix_to_feature(matrix):
    
    features = np.zeros((matrix.shape[0], int(((126 * 126) -126) / 2)))
    
    for trial in range(matrix.shape[0]):
        
        feature = np.array([])
        
        for ch1 in range(125):
            
            feature = np.append(feature, matrix[trial, ch1, ch1 +1:])
            
        features[trial, :] = feature
        
    return features

In [5]:
def combine_band_time_method(sub, methods, bands, times):
    
    Band_name = np.array(["Delta", "Theta", "Alpha", "Beta"])
    Time_name = np.array(["0_5000", "1000_5000", "0_750"])
    
    _ , labels, _ = path.load_data(sub , 'Imagination')
    Features = np.empty([labels.shape[0], 0])
    
    for method in methods:
        
        temp = 'Mat_conn_' + method.lower()
        data = path.load_matrix(sub, method)[temp]
        
        for band in bands:
            
            b_count = np.where(Band_name == band)[0][0]
            
            for time in times:
                
                t_count = np.where(Time_name == time)[0][0]
                
                Feature = matrix_to_feature(data[b_count, t_count, :, :, :])
                Features = np.append(Features, Feature, axis= 1)
    
    return Features

In [ ]:
def TDA_feature(Diagrams, Mode = 'default'):
    
    feature_extractor = TopologicalFeaturesExtractor([HolesNumberFeature(), MaxHoleLifeTimeFeature(), RelevantHolesNumber(), AverageHoleLifetimeFeature(),
                                    SumHoleLifetimeFeature(), PersistenceEntropyFeature(), BettiNumbersSumFeature(), RadiusAtMaxBNFeature()])
    
    Amp1 = Amplitude(metric='landscape', n_jobs=-1)
    Amp2 = Amplitude(metric='betti', n_jobs=-1)
    
    PL = PersistenceLandscape(n_bins = 100, n_jobs=-1)
    BC = BettiCurve(n_bins = 100, n_jobs=-1)
    
    feature1 = feature_extractor.fit_transform(Diagrams)
    
    # Remove redundant features with identical values across samples.
    # This step reduces duplicated information before classification.
    
    feature1 = feature1[:, ~np.all(feature1[1:] == feature1[:-1], axis=0)]
    
    feature2 = Amp1.fit_transform(Diagrams)[:,1].reshape(-1,1)
    feature3 = Amp2.fit_transform(Diagrams)[:,1].reshape(-1,1)
    
    PL_Xt0 = PL.fit_transform(X = Diagrams)
    BC_Xt0 = BC.fit_transform(X = Diagrams)
    
    # Select the feature representation used for classification.
    # 'All' combines the diagram-based and amplitude-based features.

    if Mode == 'default':
        
        Features = np.concatenate([feature1, feature2, feature3], axis = 1)
        
    elif Mode == 'PL':
        
        Features = np.concatenate([PL_Xt0[:,0,:], PL_Xt0[:,1,:]], axis = 1)
    
    elif Mode == 'BC':
        
        Features = np.concatenate([BC_Xt0[:,0,:], BC_Xt0[:,1,:]], axis = 1)
    
    elif Mode == 'All':
        
        Features = np.concatenate([feature1, feature2, feature3, PL_Xt0[:,0,:], PL_Xt0[:,1,:], BC_Xt0[:,0,:], BC_Xt0[:,1,:]], axis = 1)
    
    return Features

In [7]:
def combine_band_time_method_TDA(sub, methods, bands, times, Mode = 'default'):
    
    Band_name = np.array(["Delta", "Theta", "Alpha", "Beta"])
    Time_name = np.array(["0_5000", "1000_5000", "0_750"])
    
    _ , labels, _ = path.load_data(sub , 'Imagination')
    Features = np.empty([labels.shape[0], 0])
    
    for method in methods:
        
        temp = 'Mat_conn_' + method.lower()
        data = path.load_matrix(sub, method)[temp]
        
        if method == 'Pearson' or method == 'Spearman':
            data = abs(data)
        
        data = 1 - data
        
        for band in bands:
            
            b_count = np.where(Band_name == band)[0][0]
            
            for time in times:
                
                t_count = np.where(Time_name == time)[0][0]
                
                VR = VietorisRipsPersistence(metric="precomputed",  n_jobs = -1)
                Diagrams = VR.fit_transform(data[b_count, t_count, :, :, :])  
                Feature = TDA_feature(Diagrams, Mode)
                Features = np.append(Features, Feature, axis= 1)
    
    return Features

In [8]:
def combine_name(Names):
    
    Name = ''
    for name in Names:
        Name = Name + name  + '_'
    Name = Name[:-1]
    
    return Name

In [5]:
# Save classification results for subsequent statistical analysis.

def Save_df(df, type, method):
    
    df.to_excel(RESULTS_DIR / f"Classification_{type}_{method}.xlsx", index=True)

# Classification

## Imagination


### MVPA

In [ ]:
window = 20
last = 1100
stride = 1

times = round((last - window) / stride)

Model = 'RF_MVPA' 

df = pd.DataFrame(columns=['Subject', 'Window', 'Time', 'Model', 'acc_test_mean', 'CI'])

for sub in tqdm_notebook([7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35], desc= 'Subject'):
    
    data , labels, _ = path.load_data(sub , 'Imagination')
    
    for time in tqdm_notebook(range(times + 1), desc= 'Windows'):
        
        Features = np.mean(data[:, 101 + time * stride :101 + window + time * stride, :], axis=1).T
        
        acc_test_mean, CI = evaluate_model(Model, Features, labels , repeat = 10)
        
        df.loc[len(df.index)] = [sub, time , ((101 + time * stride) * 5 - 1000, (101 + window + time * stride) * 5 - 1000) , Model ,acc_test_mean, CI]
        
        Save_df(df, 'Imagination', 'MVPA_All')
        

### Connectivity Matrix

In [ ]:
Model = ['RF_FC']
methods = [['PLV'], ['PLI'], ['Pearson'], ['Spearman']]
Bands = [['Delta'], ['Theta'], ['Alpha'], [ 'Beta']]
Times = [['0_5000']]


df = pd.DataFrame(columns=['Subject', 'Band', 'Time', 'Methods', 'Model', 'acc_test_mean', 'CI'])

for time in Times:
    
    for method in tqdm_notebook(methods, desc= 'Method'):
        
        for band in tqdm_notebook(Bands, desc= 'Band'):
            
            for sub in tqdm_notebook([7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35], desc= 'Subject'):
                
                _ , labels, _ = path.load_data(sub , 'Imagination')
                
                Features = combine_band_time_method(sub, method, band, time)
                
                acc_test_mean, CI = evaluate_model(Model[0], Features, labels , repeat = 10)
                
                df.loc[len(df.index)] = [sub, combine_name(band), combine_name(time), combine_name(method) , combine_name(Model) ,acc_test_mean, CI]
                
                Save_df(df, 'Imagination', 'All_0_5000_' + Model[0] + '_Matrix')

### TDA on Connectivity Matrix

In [ ]:
Model = ['RF_TDA']
methods = [['PLV'], ['PLI'], ['Pearson'], ['Spearman']]
Bands = [['Delta'], ['Theta'], ['Alpha'], [ 'Beta']]
Times = [['0_5000']]

df = pd.DataFrame(columns=['Subject', 'Band', 'Time', 'Methods', 'Model', 'acc_test_mean', 'CI'])

for time in Times:
    
    for method in tqdm_notebook(methods, desc= 'Method'):
        
        for band in tqdm_notebook(Bands, desc= 'Band'):
            
            for sub in tqdm_notebook([7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35], desc= 'Subject'):
                
                _ , labels, _ = path.load_data(sub , 'Imagination')
                
                Features = combine_band_time_method_TDA(sub, method, band, time, Mode = 'All')
                
                acc_test_mean, CI = evaluate_model(Model[0], Features, labels , repeat = 10)
                
                df.loc[len(df.index)] = [sub, combine_name(band), combine_name(time), combine_name(method) , combine_name(Model) ,acc_test_mean, CI]
                
                Save_df(df, 'Imagination', 'All_0_5000_' + Model[0] + '_Final')